# FT05 — teste pessoal do adaptador no Colab Pro

Este notebook faz somente inferência com o adaptador salvo no seu Drive. O computador abre o navegador; a GPU é a do Colab. Nada é instalado, treinado ou montado por padrão.

**Sequência:** 1. conferir GPU; 2. obter código e rodar testes; 3. montar Drive e verificar hashes; 4. testar mensagens fictícias novas. Use uma pasta de saída nova por execução. Resultados no Drive são privados. Ao compartilhar uma cópia do notebook, apague saídas e não inclua mensagens de clientes, caminhos privados, pesos ou tokens.

O piloto sintético não está validado para atendimento real. Os casos abaixo são uma inspeção pessoal, sem rótulos de referência e sem métrica independente. O roteiro está em COLAB-TEST-PLAN.md na branch do projeto.


In [ ]:
from pathlib import Path
import shutil, subprocess, sys

# Ligue somente a etapa que está pronto para executar e rode as células na ordem.
FETCH_SOURCE = False
MOUNT_DRIVE = False
RUN_PERSONAL_TEST = False
FREEZE_REVIEWED = False
INSTALL_DEPS = False
RUN_TRAINING = False

REPO_REF = 'feat/ft05-independent-eval'
REPO_DIR = Path('/content/radar-model-router-ft05-personal')
ROOT = Path('/content/drive/MyDrive/FT05/piloto-auto-001')
RUN_DIR = ROOT / 'run-002-g4'
CORPUS = ROOT / 'synthetic-machine-checked.jsonl'
FROZEN = ROOT / 'frozen'
OUTPUT_DIR = ROOT / 'personal-smoke-001'  # escolha outro nome para uma nova tentativa

# Edite apenas textos fictícios. Não há rótulos nem respostas esperadas.
MESSAGES = [
    {'id': 'meu-01', 'text': 'A etiqueta do pacote ficou apagada e não há data de entrega definida.'},
    {'id': 'meu-02', 'text': 'A etiqueta do pacote ficou apagada e a entrega precisa ocorrer hoje.'},
    {'id': 'meu-03', 'text': 'Quero cancelar um pedido em separação sem horário de despacho anunciado.'},
    {'id': 'meu-04', 'text': 'Quero cancelar antes da renovação automática que acontece hoje.'},
    {'id': 'meu-05', 'text': 'O perfil apresenta erro ao alterar minhas preferências, sem prazo imediato.'},
    {'id': 'meu-06', 'text': 'Há uma sessão desconhecida ativa na minha conta agora.'},
    {'id': 'meu-07', 'text': 'Como posso consultar as opções de pagamento para uma compra futura?'},
    {'id': 'meu-08', 'text': 'Como vejo os prazos gerais de entrega antes de comprar?'},
]
print('Configuração pronta. Ações externas desligadas.')


## 1. Conferir o ambiente

A GPU selecionada e a GPU realmente disponível podem divergir. Esta célula só lê a configuração. Na UI do Colab, confira saldo/unidades antes da inferência.


In [ ]:
print('Python:', sys.version.split()[0])
gpu_tool = shutil.which('nvidia-smi')
if gpu_tool:
    result = subprocess.run([gpu_tool, '--query-gpu=name,memory.total',
                             '--format=csv,noheader'], capture_output=True, text=True)
    print('GPU:', result.stdout.strip() if result.returncode == 0 else result.stderr.strip())
else:
    print('GPU: não detectada; selecione um runtime com GPU antes de continuar')
# A importação de Unsloth deve preceder Torch na célula de inferência.
# A presença da GPU aqui é verificada por nvidia-smi; Torch será testado depois.


## 2. Obter código público e rodar testes

Leia a branch no GitHub antes de ativar FETCH_SOURCE. O notebook registra o commit efetivo; uma publicação estável deverá usar um tag revisado. Um clone existente não será sobrescrito. Os testes não precisam de GPU.


In [ ]:
if FETCH_SOURCE:
    assert not REPO_DIR.exists(), 'Pasta já existe; preserve a revisão ou inicie sessão limpa'
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF,
                    'https://github.com/ofelipepeixoto/radar-model-router.git',
                    str(REPO_DIR)], check=True)
    code_sha = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
    ft05_dir = REPO_DIR / 'experiments' / 'ft05'
    print('Commit do código:', code_sha)
    subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
                   cwd=ft05_dir, check=True)
else:
    print('Clone desativado. Ative FETCH_SOURCE na configuração quando revisar a branch.')


## 3. Montar o Drive e conferir artefatos

Monte apenas seu Drive e confira a pasta do piloto. O pré-voo exige corpus, split, manifesto, pesos e versões exatas da execução. Se o runtime foi recriado e faltarem dependências, resolva a instalação na cópia do notebook FT05 original antes de voltar aqui. Não troque a revisão do modelo ou o adaptador para contornar uma falha.


In [ ]:
if MOUNT_DRIVE:
    assert REPO_DIR.is_dir(), 'Execute primeiro a etapa de código'
    from google.colab import drive
    drive.mount('/content/drive')
    sys.path.insert(0, str(REPO_DIR / 'experiments' / 'ft05'))
    from personal_smoke import preflight, validate_messages
    validate_messages(MESSAGES)
    checked = preflight(RUN_DIR, CORPUS, FROZEN, versions=True)
    print('Adaptador e corpus íntegros. Modelo:', checked['meta']['model'])
    print('Revisão da base:', checked['meta']['revision'])
    print('Corpus SHA-256:', checked['corpus_sha256'])
    print('Split SHA-256:', checked['frozen_manifest_sha256'])
    print('Mensagens fictícias:', len(MESSAGES))
else:
    print('Drive não montado. Ative MOUNT_DRIVE na configuração após revisar esta célula.')


## 4. Testar mensagens novas na GPU do Colab

Ative RUN_PERSONAL_TEST só após o pré-voo. O modelo é carregado uma vez, sem treino. As previsões e latências ficam em uma pasta nova no Drive. Se a pasta já existir, escolha outro nome para preservar a execução anterior.


In [ ]:
if RUN_PERSONAL_TEST:
    assert MOUNT_DRIVE and REPO_DIR.is_dir(), 'Conclua código e Drive primeiro'
    assert not (FREEZE_REVIEWED or INSTALL_DEPS or RUN_TRAINING)
    from personal_smoke import run
    report = run(MESSAGES, RUN_DIR, OUTPUT_DIR, corpus=CORPUS, frozen=FROZEN)
    for row in report['results']:
        print(row['id'], row['parsed'] if row['parsed'] is not None
              else f"JSON inválido: {row['predicted_text']!r}",
              f"{row['latency_ms']} ms")
    print('JSON válido:', report['valid_json'], '/', report['n'])
    print('Arquivos privados:', OUTPUT_DIR / 'manifest.json', OUTPUT_DIR / 'report.json')
else:
    print('Inferência desligada. Ative RUN_PERSONAL_TEST após concluir o pré-voo.')


## 5. Interface no navegador do PC (opcional)

Esta tela usa a GPU do Colab e o adaptador privado já verificado. Ative `LAUNCH_INTERFACE` somente após a etapa 3 e execute esta célula uma vez. Digite apenas textos fictícios. Cada clique reaproveita o modelo carregado; nenhuma mensagem é enviada a servidor público ou gravada pelo código da interface. Ao reiniciar o runtime, repita configuração, código, Drive e dependências. Encerre a sessão quando terminar.


In [ ]:
LAUNCH_INTERFACE = False  # mude para True depois do pré-voo
if LAUNCH_INTERFACE:
    assert MOUNT_DRIVE and REPO_DIR.is_dir(), 'Conclua código e Drive primeiro'
    assert not (FREEZE_REVIEWED or INSTALL_DEPS or RUN_TRAINING)
    from personal_ui import launch
    panel = launch(RUN_DIR, CORPUS, FROZEN)
else:
    print('Interface desligada. Ative LAUNCH_INTERFACE após o pré-voo.')


## 6. Interpretar e encerrar

Leia cada par de mensagens: o prazo, e não apenas o problema, deve influenciar a urgência. Registre saídas inválidas e classificações questionáveis sem alterar o teste congelado ou retreinar. Este conjunto foi escolhido depois de observar erros e não estima qualidade real.

Confirme que os dois arquivos privados do teste estão legíveis no Drive e encerre o runtime do Colab. Para avaliar atendimento real, siga INDEPENDENT-EVAL.md: fonte autorizada anonimizada, dois revisores cegos e adjudicação. Não publique mensagens privadas nem o adaptador deste Drive.
